In [1]:
import os
from dotenv import load_dotenv

# 1. CORE IMPORTS (Matching your previous successful project)
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# --------------------------------------------------
# 1. CONFIGURATION & ENVIRONMENT SETUP
# --------------------------------------------------
load_dotenv()
# Using the Chroma DB path we created in Notebook 02
DB_PATH = "../db" 

# --------------------------------------------------
# 2. CORE FUNCTIONS
# --------------------------------------------------

def load_university_db():
    """
    Loads the Chroma index using local HuggingFace Embeddings.
    """
    print("📂 Loading University Vector Database...")
    
    embeddings = HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2",
        model_kwargs={'device': 'cpu'}
    )

    # Load from the local directory we saved in Notebook 02
    vector_db = Chroma(
        persist_directory=DB_PATH,
        embedding_function=embeddings
    )
    
    return vector_db

def build_university_rag_chain(vector_db):
    """
    Builds the RAG pipeline using LCEL (No legacy 'chains' needed).
    """
    print("🤖 Initializing Google Gemini for University Assistance...")

    # 1. Initialize Gemini LLM (Updated to flash for speed)
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0.2,
        top_p=0.9
    )

    # 2. Custom University Prompt Template
    prompt = ChatPromptTemplate.from_template("""
You are a professional AI Assistant for Al-Farabi Kazakh National University (KazNU). 
Your goal is to provide accurate information based ONLY on the provided document context.

If the answer is not in the context, politely state that you do not have that specific information 
and suggest the student contact the 'Keremet' Student Service Center.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
""")

    # 3. Create Retriever (Fetching top 4 relevant chunks)
    retriever = vector_db.as_retriever(search_kwargs={"k": 4})

    # 4. Helper function to format retrieved documents
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    # 5. Build RAG Chain using LCEL (The NeuralTranscript Way)
    rag_chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough()
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    return rag_chain

# --------------------------------------------------
# 3. EXECUTION FOR SUPERVISOR DEMO
# --------------------------------------------------

if __name__ == "__main__":
    # Load DB
    db = load_university_db()
    
    # Build Chain
    university_bot = build_university_rag_chain(db)
    
    # Test Query: Academic Mobility (from Policy PDF)
    user_query = "What are the requirements for participating in academic mobility?"
    
    print(f"\n❓ Student Question:\n{user_query}")
    print("\n⏳ Searching documents and generating answer...\n")
    
    # Invoke
    response = university_bot.invoke(user_query)
    
    print("✨ AI UNIVERSITY RESPONSE:\n")
    print(response)

d:\RAG_StudentSupport_Project\agentic-rag-edu-4th\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📂 Loading University Vector Database...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5831.48it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\DELL\AppData\Local\Temp\ipykernel_2688\3965126861.py:36: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(


🤖 Initializing Google Gemini for University Assistance...

❓ Student Question:
What are the requirements for participating in academic mobility?

⏳ Searching documents and generating answer...

✨ AI UNIVERSITY RESPONSE:

For participation in academic mobility, students are required to operate within the framework of interuniversity agreements/contracts and/or joint international programs/projects.

Specifically:
*   For internal academic mobility, a tripartite student agreement signed by the sending and receiving Higher Education Institutions (HEIs) is required.
*   For international academic mobility, an invitation is required.
